RL

In [ ]:
import numpy as np
import random

# ------------------------------
# 1. 迷宫环境参数定义
# ------------------------------
GRID_SIZE = 3                 # 迷宫尺寸：3x3网格
ACTIONS = [0, 1, 2, 3]        # 动作空间：0=上，1=下，2=左，3=右
START_STATE = (0, 0)          # 起点坐标
END_STATE = (2, 2)            # 终点坐标
WALL_STATE = (1, 1)           # 墙壁坐标（智能体撞墙会受到惩罚）

# ------------------------------
# 2. 核心函数定义
# ------------------------------
def init_q_table():
    """初始化Q表：记录每个状态-动作对的价值（未来累积奖励期望），初始值全为0"""
    return np.zeros((GRID_SIZE, GRID_SIZE, len(ACTIONS)))

def choose_action(state, q_table, epsilon):
    """
    使用ε-greedy策略选择动作：以概率ε随机探索，否则选择当前Q值最大的动作（利用）
    """
    row, col = state
    if random.random() < epsilon:
        return random.choice(ACTIONS)  # 随机探索
    else:
        q_values = q_table[row][col]
        max_q = np.max(q_values)
        max_actions = [a for a in ACTIONS if q_values[a] == max_q]
        return random.choice(max_actions)  # 从最优动作中随机选择一个（避免重复路径）

def take_action(state, action):
    """
    执行动作，返回下一状态和即时奖励
    - 撞墙：惩罚-1，状态不变
    - 到达终点：奖励+10
    - 普通移动：奖励0
    """
    row, col = state
    # 定义动作对应的行列变化（上、下、左、右）
    dr_dc = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    dr, dc = dr_dc[action]
    next_row = max(0, min(GRID_SIZE-1, row + dr))
    next_col = max(0, min(GRID_SIZE-1, col + dc))
    next_state = (next_row, next_col)

    if next_state == WALL_STATE:
        return state, -1   # 撞墙，返回原状态并惩罚
    elif next_state == END_STATE:
        return next_state, 10  # 到达终点，高奖励
    else:
        return next_state, 0   # 普通移动，无奖励

def train_q_learning(q_table, episodes=1000, alpha=0.1, gamma=0.9, epsilon=0.5, epsilon_decay=0.001, min_epsilon=0.1):
    """
    Q-learning训练循环：
    - episodes: 训练轮数
    - alpha: 学习率（控制Q值更新幅度）
    - gamma: 折扣因子（衡量未来奖励的重要性）
    - epsilon: 探索率（初始随机动作概率）
    - epsilon_decay: 探索率衰减系数（随着训练逐步降低探索）
    - min_epsilon: 最小探索率（保持一定探索避免局部最优）
    """
    for episode in range(episodes):
        state = START_STATE
        done = False
        while not done:
            action = choose_action(state, q_table, epsilon)
            next_state, reward = take_action(state, action)
            row, col = state
            n_row, n_col = next_state

            # 获取当前Q值和下一状态的最大Q值
            q_current = q_table[row][col][action]
            q_next_max = np.max(q_table[n_row][n_col])

            # Q值更新公式：Q(s,a) = Q(s,a) + α * [r + γ * maxQ(s',a') - Q(s,a)]
            q_table[row][col][action] = q_current + alpha * (reward + gamma * q_next_max - q_current)

            state = next_state
            if state == END_STATE:
                done = True

        # 逐步降低探索率（随着训练推进，越来越依赖已知信息）
        epsilon = max(min_epsilon, epsilon * (1 - epsilon_decay))

    return q_table

def test_optimal_path(q_table):
    """测试训练后的Q表：从起点到终点选择每一步Q值最大的动作，输出最优路径"""
    path = [START_STATE]
    state = START_STATE
    while state != END_STATE:
        row, col = state
        action = np.argmax(q_table[row][col])  # 选择当前状态Q值最大的动作
        next_state, _ = take_action(state, action)
        path.append(next_state)
        state = next_state
    return path

# ------------------------------
# 3. 主程序执行
# ------------------------------
if __name__ == "__main__":
    # 初始化Q表
    q_table = init_q_table()
    print("初始化Q表（所有Q值均为0）：")
    print(q_table)

    # 训练Q-learning模型
    print("\n开始训练...")
    trained_q_table = train_q_learning(
        q_table,
        episodes=1000,
        alpha=0.1,
        gamma=0.9,
        epsilon=0.5,
        epsilon_decay=0.001,
        min_epsilon=0.1
    )
    print("训练完成！")

    # 输出训练后关键状态的Q值
    print("\n训练后关键状态的Q值：")
    print(f"起点(0,0)的Q值: {trained_q_table[0][0].round(2)}")
    print(f"墙壁旁(0,1)的Q值: {trained_q_table[0][1].round(2)}")
    print(f"终点前(2,1)的Q值: {trained_q_table[2][1].round(2)}")

    # 测试并输出最优路径
    print("\n生成最优路径...")
    path = test_optimal_path(trained_q_table)
    print("最优路径: " + " → ".join([str(s) for s in path]))

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# ------------------------------
# 1. 迷宫环境参数定义
# ------------------------------
GRID_SIZE = 3                 # 迷宫尺寸：3x3网格
ACTIONS = [0, 1, 2, 3]        # 动作空间：0=上，1=下，2=左，3=右
ACTION_SYMBOLS = ['↑', '↓', '←', '→']  # 动作符号
START_STATE = (0, 0)          # 起点坐标
END_STATE = (2, 2)            # 终点坐标
WALL_STATE = (1, 1)           # 墙壁坐标

# ------------------------------
# 2. 核心函数定义 (与之前相同)
# ------------------------------
def init_q_table():
    """初始化Q表"""
    return np.zeros((GRID_SIZE, GRID_SIZE, len(ACTIONS)))

def choose_action(state, q_table, epsilon):
    """ε-greedy策略选择动作"""
    row, col = state
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    else:
        q_values = q_table[row][col]
        max_q = np.max(q_values)
        max_actions = [a for a in ACTIONS if q_values[a] == max_q]
        return random.choice(max_actions)

def take_action(state, action):
    """执行动作，返回下一状态和即时奖励"""
    row, col = state
    dr_dc = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    dr, dc = dr_dc[action]
    next_row = max(0, min(GRID_SIZE-1, row + dr))
    next_col = max(0, min(GRID_SIZE-1, col + dc))
    next_state = (next_row, next_col)

    if next_state == WALL_STATE:
        return state, -1   # 撞墙
    elif next_state == END_STATE:
        return next_state, 10  # 到达终点
    else:
        return next_state, 0

# ------------------------------
# 3. 增强的训练函数 (添加数据记录)
# ------------------------------
def train_q_learning_enhanced(q_table, episodes=1000, alpha=0.1, gamma=0.9, 
                             epsilon=0.5, epsilon_decay=0.001, min_epsilon=0.1):
    """增强的训练函数，记录训练过程数据"""
    episode_rewards = []      # 每轮的累计奖励
    episode_steps = []        # 每轮的步数
    exploration_rates = []   # 每轮的探索率
    q_table_history = []      # 每100轮保存一次Q表快照
    
    for episode in range(episodes):
        state = START_STATE
        done = False
        total_reward = 0
        steps = 0
        
        while not done:
            action = choose_action(state, q_table, epsilon)
            next_state, reward = take_action(state, action)
            row, col = state
            n_row, n_col = next_state

            # Q值更新
            q_current = q_table[row][col][action]
            q_next_max = np.max(q_table[n_row][n_col])
            q_table[row][col][action] = q_current + alpha * (reward + gamma * q_next_max - q_current)

            state = next_state
            total_reward += reward
            steps += 1
            
            if state == END_STATE:
                done = True

        # 记录本轮数据
        episode_rewards.append(total_reward)
        episode_steps.append(steps)
        exploration_rates.append(epsilon)
        
        # 每100轮保存Q表快照
        if episode % 100 == 0:
            q_table_history.append(q_table.copy())
        
        # 衰减探索率
        epsilon = max(min_epsilon, epsilon * (1 - epsilon_decay))
        
        # 每100轮打印进度
        if episode % 100 == 0:
            print(f"Episode {episode}: Reward={total_reward}, Steps={steps}, Epsilon={epsilon:.3f}")

    return q_table, episode_rewards, episode_steps, exploration_rates, q_table_history

def test_optimal_path(q_table):
    """测试训练后的Q表，输出最优路径"""
    path = [START_STATE]
    state = START_STATE
    while state != END_STATE:
        row, col = state
        action = np.argmax(q_table[row][col])
        next_state, _ = take_action(state, action)
        path.append(next_state)
        state = next_state
    return path

# ------------------------------
# 4. 可视化函数
# ------------------------------
def plot_training_process(rewards, steps, exploration_rates):
    """绘制训练过程曲线"""
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12))
    
    # 奖励曲线
    ax1.plot(rewards)
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.set_title('Training Rewards')
    ax1.grid(True)
    
    # 平滑奖励曲线 (移动平均)
    window_size = 50
    smooth_rewards = np.convolve(rewards, np.ones(window_size)/window_size, mode='valid')
    ax1.plot(range(window_size-1, len(rewards)), smooth_rewards, 'r-', linewidth=2, label=f'Smooth ({window_size}-episode avg)')
    ax1.legend()
    
    # 步数曲线
    ax2.plot(steps)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Steps')
    ax2.set_title('Steps per Episode')
    ax2.grid(True)
    
    # 探索率曲线
    ax3.plot(exploration_rates)
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Exploration Rate (ε)')
    ax3.set_title('Exploration Rate Decay')
    ax3.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_process.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_q_table_heatmap(q_table, episode=None):
    """绘制Q表热力图"""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # 计算每个状态的最高Q值
    max_q_values = np.max(q_table, axis=2)
    
    # 创建热力图
    im = ax.imshow(max_q_values, cmap='viridis', interpolation='nearest')
    
    # 添加颜色条
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel('Max Q-Value', rotation=-90, va="bottom")
    
    # 设置刻度
    ax.set_xticks(np.arange(GRID_SIZE))
    ax.set_yticks(np.arange(GRID_SIZE))
    
    # 在每个单元格中显示Q值信息
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            # 显示最大Q值
            text = ax.text(j, i, f"{max_q_values[i, j]:.1f}",
                          ha="center", va="center", color="w", fontweight='bold')
            
            # 显示最佳动作方向
            best_action = np.argmax(q_table[i, j])
            action_symbol = ACTION_SYMBOLS[best_action]
            ax.text(j, i+0.3, action_symbol, ha="center", va="center", color="red", fontsize=16)
    
    # 标记特殊格子
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            if (i, j) == START_STATE:
                ax.text(j, i-0.3, "START", ha="center", va="center", color="green", fontweight='bold')
            elif (i, j) == END_STATE:
                ax.text(j, i-0.3, "GOAL", ha="center", va="center", color="blue", fontweight='bold')
            elif (i, j) == WALL_STATE:
                ax.text(j, i-0.3, "WALL", ha="center", va="center", color="red", fontweight='bold')
    
    title = "Q-Table Heatmap" + (f" (Episode {episode})" if episode is not None else "")
    ax.set_title(title)
    plt.savefig(f'q_table_heatmap{f"_{episode}" if episode else ""}.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_optimal_path(path, q_table):
    """绘制最优路径"""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 创建网格
    grid = np.zeros((GRID_SIZE, GRID_SIZE))
    
    # 标记特殊格子
    grid[START_STATE] = 0.2    # 起点
    grid[END_STATE] = 0.4      # 终点
    grid[WALL_STATE] = 0.6     # 墙壁
    
    # 显示网格
    im = ax.imshow(grid, cmap='Pastel1', interpolation='nearest')
    
    # 绘制路径
    for k in range(len(path)-1):
        i1, j1 = path[k]
        i2, j2 = path[k+1]
        ax.arrow(j1, i1, (j2-j1)*0.8, (i2-i1)*0.8, head_width=0.2, 
                head_length=0.2, fc='blue', ec='blue', linewidth=2)
    
    # 添加格子标注
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            if (i, j) == START_STATE:
                ax.text(j, i, "START", ha="center", va="center", color="green", fontweight='bold')
            elif (i, j) == END_STATE:
                ax.text(j, i, "GOAL", ha="center", va="center", color="blue", fontweight='bold')
            elif (i, j) == WALL_STATE:
                ax.text(j, i, "WALL", ha="center", va="center", color="red", fontweight='bold')
            else:
                ax.text(j, i, f"({i},{j})", ha="center", va="center", color="black")
    
    ax.set_title("Optimal Path Found by Q-Learning")
    plt.savefig('optimal_path.png', dpi=300, bbox_inches='tight')
    plt.show()

def create_training_animation(q_table_history, rewards):
    """创建训练过程动画"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 初始化动画帧
    def init():
        ax1.clear()
        ax2.clear()
        return ax1, ax2
    
    # 动画更新函数
    def update(frame):
        ax1.clear()
        ax2.clear()
        
        # 左图：Q表热力图
        q_table = q_table_history[frame]
        episode = frame * 100
        
        max_q_values = np.max(q_table, axis=2)
        im = ax1.imshow(max_q_values, cmap='viridis', interpolation='nearest')
        
        for i in range(GRID_SIZE):
            for j in range(GRID_SIZE):
                best_action = np.argmax(q_table[i, j])
                action_symbol = ACTION_SYMBOLS[best_action]
                ax1.text(j, i, f"{max_q_values[i, j]:.1f}\n{action_symbol}", 
                        ha="center", va="center", color="white" if max_q_values[i, j] < 5 else "black",
                        fontweight='bold')
        
        ax1.set_title(f"Q-Table Heatmap (Episode {episode})")
        
        # 右图：奖励曲线
        ax2.plot(rewards[:episode+1])
        ax2.axvline(x=episode, color='r', linestyle='--', alpha=0.5)
        ax2.set_xlabel('Episode')
        ax2.set_ylabel('Total Reward')
        ax2.set_title('Training Rewards')
        ax2.grid(True)
        
        # 添加当前奖励值标注
        if episode < len(rewards):
            ax2.text(episode, rewards[episode], f'{rewards[episode]:.1f}', 
                    ha='left', va='bottom')
        
        return ax1, ax2
    
    # 创建动画
    ani = FuncAnimation(fig, update, frames=len(q_table_history),
                        init_func=init, blit=False, repeat=False)
    
    # 保存动画
    ani.save('q_learning_training.gif', writer='pillow', fps=2, dpi=100)
    
    plt.tight_layout()
    plt.show()
    
    return ani

# ------------------------------
# 5. 主程序执行
# ------------------------------
if __name__ == "__main__":
    # 初始化Q表
    q_table = init_q_table()
    print("开始训练Q-learning模型...")
    
    # 训练模型并记录数据
    trained_q_table, rewards, steps, exploration_rates, q_table_history = train_q_learning_enhanced(
        q_table,
        episodes=1000,
        alpha=0.1,
        gamma=0.9,
        epsilon=0.5,
        epsilon_decay=0.001,
        min_epsilon=0.1
    )
    print("训练完成！")
    
    # 测试最优路径
    print("\n生成最优路径...")
    path = test_optimal_path(trained_q_table)
    print("最优路径:", " → ".join([str(s) for s in path]))
    
    # 可视化训练过程
    print("\n绘制训练过程曲线...")
    plot_training_process(rewards, steps, exploration_rates)
    
    print("绘制最终Q表热力图...")
    plot_q_table_heatmap(trained_q_table)
    
    print("绘制最优路径图...")
    plot_optimal_path(path, trained_q_table)
    
    print("创建训练过程动画...")
    create_training_animation(q_table_history, rewards)
    
    print("所有可视化已完成！检查当前目录下的PNG和GIF文件")